# SLAE-CS and SLAE-ERR on a Colab GPU

Installs SLAE-CS and SLAE-ERR, checks the installation against the reference output of the paper's environment, and
predicts backbone amide ¹⁵N chemical shifts (SLAE-CS, 10-seed ensemble) and their expected error
(SLAE-ERR) for a protein structure.

**Runtime → Change runtime type → GPU** (a T4 is enough). The install takes a few minutes.

**Browser:** use Chrome or Firefox. In Safari, Colab's upload button fails (`RangeError: Maximum call stack size
exceeded`); there, put your structures in place by hand before section 3 (see section 3).

Code: [WaymentSteeleLab/SLAE-CS-and-SLAE-ERR](https://github.com/WaymentSteeleLab/SLAE-CS-and-SLAE-ERR) · weights:
[hws-lab/SLAE-CS-and-SLAE-ERR](https://huggingface.co/hws-lab/SLAE-CS-and-SLAE-ERR)

## 1. Code and environment

In [ ]:
!nvidia-smi -L

In [ ]:
import os, subprocess

REPO = "https://github.com/WaymentSteeleLab/SLAE-CS-and-SLAE-ERR.git"
CODE = "/content/SLAE-CS"
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", REPO, CODE], check=True)
%cd {CODE}
!git log -1 --format='%h %s'

In [ ]:
# A separate Python 3.11 environment with the versions of the paper's environment
# (containers/slae/pip_list.txt), CUDA 12.4 builds. Colab's own Python is newer than these packages support.
# python-dotenv: needed by atomworks 1.0.3 but not declared by it.
!pip install -q uv
!uv venv -q --seed --python 3.11 /content/slae-env
PY = "/content/slae-env/bin/python"
!{PY} -m pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
!{PY} -m pip install -q torch-scatter==2.1.2 torch-cluster==1.6.3 torch-sparse==0.6.18 -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!{PY} -m pip install -q torch==2.6.0 torch-geometric==2.7.0 e3nn==0.5.8 einops==0.8.1 einx==0.3.0 esm==3.2.1 \
    lightning==2.5.6 hydra-core==1.3.2 omegaconf==2.3.0 loguru==0.7.3 opt-einsum-fx==0.1.4 \
    torch-runstats==0.2.0 biotite==1.5.0 atomworks==1.0.3 python-dotenv==1.2.1 safetensors==0.6.2 \
    huggingface-hub==0.36.0 numpy==1.26.4 gemmi==0.7.5
!{PY} -c "import torch, torch_geometric; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| PyG', torch_geometric.__version__)"

## 2. Weights and installation test

In [ ]:
import glob
if not os.path.isdir("model"):
    subprocess.run([PY, "scripts/download_from_hf.py", "--no-data"], check=True)   # weights only, into model/
n = len(glob.glob("model/SLAE-CS-PANAV/seed*/model.safetensors"))
assert n == 10 and os.path.exists("model/SLAE-ERR/model.safetensors"), f"incomplete weights: {n} SLAE-CS models"
print("10 SLAE-CS models + SLAE-ERR in", os.path.abspath("model"))

In [ ]:
# predict.py on the three structures in examples/, compared with the reference output (NVIDIA A100)
!{PY} tests/smoke_test.py --device cuda

## 3. Your own structures

One protein per file: `.pdb` or `.cif` (also gzipped). Before the prediction, each file is cleaned with gemmi
(`scripts/inference_custom/clean_structures.py`): first model only; waters, ligands, ions, DNA/RNA and hydrogens
removed; PDB format.

**Chrome / Firefox:** run the cell below and select one or more `.pdb` files in the upload dialog.

**Safari** (the upload dialog fails there; this also works in any browser):
1. Open the file panel: folder icon in the left sidebar. Its top level is `/content` (with `sample_data/`).
2. Right-click in the panel on `/content` → **New folder** → name it `my_structures`.
3. Drag your `.pdb` files onto the `my_structures` folder (not onto empty space in the panel, which puts them
   in `/content`) and check that they appear inside it.
4. Run the cell below: it uses the files in `/content/my_structures` and does not open the upload dialog.

Files in the panel are lost when the runtime disconnects or restarts. To predict a different set, delete the old
files in `my_structures` first.


In [ ]:
import shutil, subprocess
from pathlib import Path

MY = Path("/content/my_structures")             # structures placed by hand (file panel) or uploaded below
CLEAN = Path("/content/clean_structures")       # what predict.py reads: cleaned PDB files
PDB = "examples/structures/15419_1.pdb"
mine = lambda: [p for p in MY.glob("*") if p.name.lower().removesuffix(".gz").endswith((".pdb", ".ent", ".cif", ".mmcif"))]
if not mine():
    try:
        from google.colab import files
    except ImportError:                         # not on Colab: keep the example
        files = None
    if files is not None:
        try:
            up = files.upload()                 # select one or more .pdb / .cif files; cancel to keep the example
        except Exception as e:
            raise SystemExit(f"Upload failed ({e}). In Safari the upload dialog does not work: use Chrome or "
                             "Firefox, or put the files into /content/my_structures with the file panel "
                             "(see above) and run this cell again.")
        MY.mkdir(exist_ok=True)
        for name in up:
            shutil.move(name, MY / name)
if mine():
    shutil.rmtree(CLEAN, ignore_errors=True)    # first model, amino acids only, PDB format
    subprocess.run([PY, "scripts/inference_custom/clean_structures.py", str(MY), "--out", str(CLEAN)], check=True)
    PDB = str(CLEAN)
print("input:", PDB)
!{PY} predict.py --weights model/SLAE-CS-PANAV --error-model model/SLAE-ERR --pdb "$PDB" --out prediction --device cuda


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("prediction.csv")
print(f"{df.protein.nunique()} protein(s), {len(df)} residues (prolines have no amide N and are left out)")
display(df[["protein", "chain", "resnum", "resname", "N_mean", "N_std", "N_abs_error"]].head(10))

for prot, d in list(df.groupby("protein"))[:5]:          # plots of the first 5 proteins; all are in prediction.csv
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
    a1.errorbar(d.resnum, d.N_mean, yerr=d.N_std, fmt="o", ms=3, lw=0.8, capsize=0)
    a1.invert_yaxis()
    a1.set_ylabel("predicted 15N (ppm)")
    a1.set_title(f"{prot} - SLAE-CS: mean of 10 seeds (bars: spread between seeds)")
    a2.bar(d.resnum, d.N_abs_error, width=0.8)
    a2.set_ylabel("expected |error| (ppm)")
    a2.set_xlabel("residue number")
    a2.set_title("SLAE-ERR")
    plt.tight_layout()
    plt.show()


In [ ]:
try:
    from google.colab import files
    files.download("prediction.csv")
    files.download("prediction.json")
except Exception:
    print("prediction.csv and prediction.json are in", os.getcwd())